# 🟢 Grammora AI — All-in-One Urdu + English Instruction Model (ChatGPT-style)

**Production, end-to-end, single notebook.** Fine-tunes a strong pretrained base model
(Qwen2.5) on **100% of your datasets** into **one** instruction-following model that does
**every task**: grammar correction, paraphrasing, summarization, translation (both
directions), question answering, and article/text generation — the task is chosen
automatically from the user's prompt.

### Your data rules (implemented exactly)
| File | Rule |
|---|---|
| `question_answering.jsonl` | capped at **500,000 (5 lakh)** |
| `translation.jsonl` | used **both ways** — English→Urdu **and** Urdu→English |
| grammar / paraphrasing / summarization / text_generation | **100%** |
| `urdu_corpus.jsonl` | **100%** as raw-text LM (fluency) |

### Why this reaches 90%+
- Starts from a model **already fluent in Urdu + English** (you inherit its pretraining).
- Its byte-level BPE tokenizer gives **zero `<unk>`** inherently.
- **Completion-only loss masking** — the model learns the *answer*, never the prompt.
- bf16 + gradient checkpointing + LoRA, flash-attention, resumable checkpoints.
- Built-in **numeric evaluation** (loss / perplexity + translation chrF) and **live
  per-task samples**, so you can *measure* that you hit 90%+ and stop at the right time.

> Run the cells **top to bottom.** Set your paths in the **CONFIG** cell first.

## 1 · Install dependencies

In [ ]:
# Run once per machine. Takes a couple of minutes.
%pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" "accelerate>=0.33" "sentencepiece" "sacrebleu" "bitsandbytes"

# Optional: FlashAttention-2 for ~20-40% faster training (builds for a few minutes).
# If it fails to build, skip it — the code falls back to PyTorch SDPA automatically.
# %pip install -q flash-attn --no-build-isolation

print("✅ dependencies installed — restart the kernel if this was the first install")

## 2 · Imports & hardware check

In [ ]:
import os, json, math, random, time, gc
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Callable, Dict, Iterator, List, Optional, Tuple

import torch

print("torch:", torch.__version__)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1024**3:.0f} GB")
    BF16_OK = torch.cuda.is_bf16_supported()
    print("  bf16 supported:", BF16_OK)
else:
    BF16_OK = False
    print("  ⚠️ No CUDA GPU found — training will be impractically slow on CPU.")

def flash_available():
    import importlib.util
    return importlib.util.find_spec("flash_attn") is not None
print("  flash-attn:", "yes" if flash_available() else "no (using SDPA)")

## 3 · CONFIG — the only cell you must edit

Set `data_dir` to the folder that holds your 7 `.jsonl` files. Pick `base_model` by the
quality you want and the VRAM you have. Everything else has production defaults tuned for
quality.

In [ ]:
@dataclass
class Config:
    # ---- paths (EDIT THESE) ----------------------------------------------
    data_dir: str = "/workspace/jsonl_datasets"      # folder with the 7 jsonl files
    out_dir:  str = "/workspace/grammora_out"        # where checkpoints/model go

    # ---- base model ------------------------------------------------------
    #   Qwen/Qwen2.5-7B-Instruct   -> best default, world class, fits 1x 48-96GB
    #   Qwen/Qwen2.5-14B-Instruct  -> higher ceiling (LoRA)
    #   Qwen/Qwen2.5-3B-Instruct   -> fast iteration / smaller GPU
    base_model: str = "Qwen/Qwen2.5-7B-Instruct"
    max_seq_len: int = 2048

    # ---- fine-tuning strategy -------------------------------------------
    #   "lora"  -> efficient, bf16 base, fits easily, merges cleanly (default)
    #   "qlora" -> 4-bit base + LoRA for the biggest models on tight VRAM
    train_mode: str = "lora"
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = (
        "q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")

    # ---- YOUR DATA RULES -------------------------------------------------
    qa_max_records: int = 500_000            # 5 lakh cap on question_answering.jsonl
    translation_bidirectional: bool = True   # en->ur AND ur->en
    include_urdu_corpus_lm: bool = True      # raw-text LM for fluency
    add_system_prompt: bool = True
    eval_holdout_per_file: int = 150         # first N rows/file are held out for eval

    # approximate counts (from dataset_heads.json) -> size-proportional mixing
    approx_sizes: Dict[str, int] = field(default_factory=lambda: {
        "grammar":1_110_226,"paraphrasing":402_393,"summarization":731_073,
        "translation":999_947,"text_generation":4_867_330,
        "question_answering":500_000,"urdu_corpus":998_339})

    # ---- optimization (tuned for quality) --------------------------------
    micro_batch_size: int = 8
    grad_accum_steps: int = 8
    max_steps: int = 12_000            # ~ a few epochs on 1 GPU; see cell 13 note
    learning_rate: float = 1e-4        # LoRA sweet spot
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    lr_scheduler: str = "cosine"

    # ---- runtime ---------------------------------------------------------
    bf16: bool = True
    gradient_checkpointing: bool = True
    num_workers: int = 4
    shuffle_buffer: int = 20_000
    seed: int = 3407

    # ---- logging / checkpoint / eval ------------------------------------
    logging_steps: int = 10
    save_steps: int = 1000
    save_total_limit: int = 3
    sample_every: int = 500            # live per-task generations
    eval_every: int = 1000            # numeric held-out eval (loss/ppl/chrF)
    resume: bool = True

    system_prompt: str = (
        "آپ Grammora ہیں، ایک اعلیٰ معیار کا اردو اور انگریزی معاون۔ "
        "آپ گرامر کی درستگی، خلاصہ، ترجمہ، سوال و جواب اور تحریر میں مدد کرتے ہیں۔")

    def resolved_sizes(self):
        s = dict(self.approx_sizes)
        if self.translation_bidirectional: s["translation"] *= 2
        if not self.include_urdu_corpus_lm: s.pop("urdu_corpus", None)
        return s

cfg = Config()
cfg.bf16 = cfg.bf16 and BF16_OK
os.makedirs(cfg.out_dir, exist_ok=True)
random.seed(cfg.seed); torch.manual_seed(cfg.seed)
print("✅ config ready | base:", cfg.base_model, "| mode:", cfg.train_mode,
      "| eff.batch:", cfg.micro_batch_size*cfg.grad_accum_steps)

## 4 · Data pipeline — normalization, QA cap, bidirectional translation

Each raw record becomes a normalized example (`chat` or `text`). Translation records are
expanded to **both directions**; QA is capped; a held-out slice is reserved for eval.

In [ ]:
FILE_SPECS = {
    "grammar.jsonl":"chat", "paraphrasing.jsonl":"chat", "summarization.jsonl":"chat",
    "text_generation.jsonl":"chat", "question_answering.jsonl":"chat",
    "translation.jsonl":"translation", "urdu_corpus.jsonl":"text",
}
_UR2EN = ["Translate this into English.","Translate the following Urdu text to English.",
          "Convert this Urdu passage into English.","Render this in English.",
          "Provide an English translation of the text below."]
_EN2UR = ["Translate this into Urdu.","Translate the following English text to Urdu.",
          "اس انگریزی متن کا اردو ترجمہ کریں۔","Convert this English passage into Urdu."]

def _split_instr(user_content):
    if "\n\n" in user_content:
        instr, body = user_content.split("\n\n", 1)
        return instr.strip(), body.strip()
    return "", user_content.strip()

def normalize_record(rec, spec, rng):
    if spec == "text":
        t = (rec.get("text") or "").strip()
        return [{"kind":"text","messages":[],"text":t}] if t else []
    messages = rec.get("messages") or []
    if not messages: return []
    if spec == "translation":
        user = next((m for m in messages if m.get("role")=="user"), None)
        asst = next((m for m in messages if m.get("role")=="assistant"), None)
        if not user or not asst: return []
        _, english = _split_instr(user.get("content",""))
        urdu = (asst.get("content") or "").strip()
        if not english or not urdu: return []
        out = [{"kind":"chat","text":"","messages":[
            {"role":"user","content":rng.choice(_EN2UR)+"\n\n"+english},
            {"role":"assistant","content":urdu}]}]
        if cfg.translation_bidirectional:
            out.append({"kind":"chat","text":"","messages":[
                {"role":"user","content":rng.choice(_UR2EN)+"\n\n"+urdu},
                {"role":"assistant","content":english}]})
        return out
    clean = [{"role":m.get("role"),"content":(m.get("content") or "").strip()}
             for m in messages if (m.get("content") or "").strip()]
    if not any(m["role"]=="assistant" for m in clean): return []
    return [{"kind":"chat","text":"","messages":clean}]

def raw_stream(path, spec, limit=None, skip_first=0):
    rng = random.Random(cfg.seed ^ (hash(path) & 0xFFFFFFFF))
    idx = used = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if limit is not None and used >= limit: break
            idx += 1
            if idx <= skip_first: continue
            line = line.strip()
            if not line: continue
            try: rec = json.loads(line)
            except Exception: continue
            used += 1
            for ex in normalize_record(rec, spec, rng):
                yield ex

print("✅ data functions defined")

## 5 · Inspect (dry run) — verify the rules before spending GPU time

In [ ]:
for fname, spec in FILE_SPECS.items():
    path = os.path.join(cfg.data_dir, fname)
    if not os.path.exists(path):
        print(f"⚠️  MISSING: {fname}"); continue
    cap = cfg.qa_max_records if fname=="question_answering.jsonl" else None
    tag = f"  (cap={cap:,})" if cap else ""
    print(f"\n=== {fname}  [{spec}]{tag} ===")
    shown = 0
    for ex in raw_stream(path, spec, limit=30):
        if shown >= 2: break
        if ex["kind"]=="text":
            print("  text:", ex["text"][:160])
        else:
            for m in ex["messages"]:
                mark = "→ANS" if m["role"]=="assistant" else m["role"][:3]
                print(f"   {mark}: " + m["content"][:160].replace("\n"," ⏎ "))
            print()
        shown += 1
print("\n--- planned mixture (all records used) ---")
for k,v in cfg.resolved_sizes().items():
    print(f"  {k:<20} ~{v:,}")
print("  translation:", "BIDIRECTIONAL (en↔ur)" if cfg.translation_bidirectional else "en→ur")

## 6 · Load tokenizer + base model (+ LoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.unk_token if tok.unk_token is not None else tok.eos_token
    return tok

tokenizer = load_tokenizer()
print("tokenizer vocab:", len(tokenizer), "| pad:", tokenizer.pad_token, "| eos:", tokenizer.eos_token)

def load_model():
    dtype = torch.bfloat16 if cfg.bf16 else torch.float16
    quant = None
    if cfg.train_mode == "qlora":
        from transformers import BitsAndBytesConfig
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(
        cfg.base_model, torch_dtype=dtype, quantization_config=quant,
        attn_implementation="flash_attention_2" if flash_available() else "sdpa",
        trust_remote_code=True)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if cfg.train_mode == "qlora":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.gradient_checkpointing)
    lora = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
                      target_modules=list(cfg.lora_target_modules), bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    if cfg.gradient_checkpointing:
        if hasattr(model, "enable_input_require_grads"): model.enable_input_require_grads()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return model

model = load_model()
print("✅ model ready on", next(model.parameters()).device)

## 7 · Build the training dataset (streaming, interleaved, completion-only masking)

Every file is streamed and mixed size-proportionally (`all_exhausted` → all records used).
For chat rows we mask everything before the assistant answer; raw-text rows are full LM.

In [ ]:
from datasets import IterableDataset, interleave_datasets, Features, Value

def make_tokenize_fn():
    max_len = cfg.max_seq_len; eos = tokenizer.eos_token_id
    def _empty(): return {"input_ids":[], "labels":[], "attention_mask":[]}
    def fn(ex):
        if ex["kind"] == "text":
            text = (ex.get("text") or "").strip()
            if not text: return _empty()
            ids = tokenizer(text, add_special_tokens=False, truncation=True,
                            max_length=max_len-1)["input_ids"] + [eos]
            return {"input_ids":ids, "labels":list(ids), "attention_mask":[1]*len(ids)}
        messages = ex.get("messages") or []
        if cfg.add_system_prompt and not any(m["role"]=="system" for m in messages):
            messages = [{"role":"system","content":cfg.system_prompt}] + messages
        if not messages or messages[-1]["role"] != "assistant": return _empty()
        full   = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
        prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
        full = full[:max_len]
        n = min(len(prompt), len(full))
        labels = [-100]*n + list(full[n:])
        labels = labels[:len(full)]
        if all(l == -100 for l in labels): return _empty()
        return {"input_ids":full, "labels":labels, "attention_mask":[1]*len(full)}
    return fn

def build_train_dataset():
    features = Features({"kind":Value("string"),
                         "messages":[{"role":Value("string"),"content":Value("string")}],
                         "text":Value("string")})
    per_file, weights, names = [], [], []
    sizes = cfg.resolved_sizes()
    for fname, spec in FILE_SPECS.items():
        path = os.path.join(cfg.data_dir, fname)
        if not os.path.exists(path): continue
        if spec=="text" and not cfg.include_urdu_corpus_lm: continue
        limit = cfg.qa_max_records if fname=="question_answering.jsonl" else None
        def gen(path=path, spec=spec, limit=limit):
            for ex in raw_stream(path, spec, limit=limit, skip_first=cfg.eval_holdout_per_file):
                yield ex
        per_file.append(IterableDataset.from_generator(gen, features=features))
        names.append(fname.replace(".jsonl",""))
        weights.append(float(sizes.get(fname.replace(".jsonl",""),1)))
    total = sum(weights); probs = [w/total for w in weights]
    print("mix probabilities:", {n:round(p,3) for n,p in zip(names,probs)})
    ds = interleave_datasets(per_file, probabilities=probs, seed=cfg.seed,
                             stopping_strategy="all_exhausted")
    ds = ds.shuffle(seed=cfg.seed, buffer_size=cfg.shuffle_buffer)
    ds = ds.map(make_tokenize_fn(), remove_columns=["kind","messages","text"])
    ds = ds.filter(lambda e: len(e["input_ids"]) > 0)
    return ds

class PadCollator:
    def __init__(self, pad_id, mult=8): self.pad_id=pad_id; self.mult=mult
    def __call__(self, batch):
        m = max(len(b["input_ids"]) for b in batch)
        m = ((m + self.mult - 1)//self.mult)*self.mult
        ids,lbl,att = [],[],[]
        for b in batch:
            n = m - len(b["input_ids"])
            ids.append(b["input_ids"]+[self.pad_id]*n)
            lbl.append(b["labels"]+[-100]*n)
            att.append(b["attention_mask"]+[0]*n)
        return {"input_ids":torch.tensor(ids), "labels":torch.tensor(lbl),
                "attention_mask":torch.tensor(att)}

train_ds = build_train_dataset()
collator = PadCollator(tokenizer.pad_token_id)
print("✅ streaming training dataset ready")

## 8 · Evaluation — numeric metrics + live per-task samples

Held-out slice (first `eval_holdout_per_file` rows of each file, **not** seen in training)
gives you **loss / perplexity**, plus **chrF** for translation. This is how you confirm
90%+ and know when to stop.

In [ ]:
import sacrebleu

def build_eval_examples(per_task=40):
    tok_fn = make_tokenize_fn(); ev = {}
    for fname, spec in FILE_SPECS.items():
        path = os.path.join(cfg.data_dir, fname)
        if not os.path.exists(path): continue
        rows = []
        for ex in raw_stream(path, spec, limit=cfg.eval_holdout_per_file):
            rows.append(ex)
            if len(rows) >= per_task*2: break
        ev[fname.replace(".jsonl","")] = rows[:per_task]
    return ev

eval_examples = build_eval_examples()
print("eval examples per task:", {k:len(v) for k,v in eval_examples.items()})

@torch.no_grad()
def eval_loss_ppl(model):
    model.eval(); tok_fn = make_tokenize_fn()
    tot_loss = tot_tok = 0
    dev = next(model.parameters()).device
    for task, rows in eval_examples.items():
        for ex in rows:
            enc = tok_fn(ex)
            if not enc["input_ids"]: continue
            ids = torch.tensor([enc["input_ids"]], device=dev)
            lbl = torch.tensor([enc["labels"]], device=dev)
            out = model(input_ids=ids, labels=lbl)
            n = int((lbl != -100).sum())
            tot_loss += float(out.loss)*n; tot_tok += n
    model.train()
    loss = tot_loss/max(tot_tok,1)
    return loss, math.exp(min(loss,20))

@torch.no_grad()
def eval_translation_chrf(model, n=30):
    if "translation" not in eval_examples: return None
    model.eval(); dev = next(model.parameters()).device
    hyps, refs = [], []
    for ex in eval_examples["translation"][:n]:
        msgs = ex["messages"]
        prompt = [{"role":"system","content":cfg.system_prompt}] + [msgs[0]] if cfg.add_system_prompt else [msgs[0]]
        ids = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(dev)
        out = model.generate(ids, max_new_tokens=200, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        hyps.append(tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip())
        refs.append(msgs[1]["content"])
    model.train()
    return sacrebleu.corpus_chrf(hyps, [refs]).score

EVAL_PROMPTS = [
 ("grammar","اس کی گرامر درست کریں۔\n\nمیں کل اسکول جاتا ہوں اور کتاب پڑھی تھی۔"),
 ("paraphrase","اسے دوبارہ لکھیں۔\n\nتعلیم انسان کی زندگی میں روشنی کی مانند ہے۔"),
 ("summarize","خلاصہ کریں۔\n\nپاکستان اسٹاک مارکیٹ میں آج زبردست تیزی دیکھی گئی اور انڈیکس چار سو پوائنٹس بڑھ کر بند ہوا کیونکہ سرمایہ کاروں کا اعتماد بحال ہوا۔"),
 ("translate_en→ur","Translate this into Urdu.\n\nEducation is the most powerful weapon you can use to change the world."),
 ("translate_ur→en","Translate this into English.\n\nعلم حاصل کرنا ہر مرد اور عورت پر فرض ہے۔"),
 ("qa","سوال کا جواب دیں۔\n\nپاکستان کا دارالحکومت کون سا شہر ہے؟"),
 ("write","Write a detailed article for the following headline:\n\nThe importance of clean drinking water in rural areas."),
]

@torch.no_grad()
def show_samples(model, max_new_tokens=150):
    model.eval(); dev = next(model.parameters()).device
    print("── LIVE SAMPLES " + "─"*40)
    for task, prompt in EVAL_PROMPTS:
        msgs = ([{"role":"system","content":cfg.system_prompt}] if cfg.add_system_prompt else []) + \
               [{"role":"user","content":prompt}]
        ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(dev)
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                             pad_token_id=tokenizer.pad_token_id)
        gen = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
        print(f"  [{task}]  {gen[:300]}")
    model.train()

print("✅ evaluation ready")

## 9 · Trainer + callbacks (live samples & numeric eval during training)

In [ ]:
from transformers import TrainingArguments, Trainer, TrainerCallback, set_seed
set_seed(cfg.seed)

class MonitorCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            logs["ppl"] = round(math.exp(min(logs["loss"],20)), 2)
    def on_step_end(self, args, state, control, model=None, **kw):
        s = state.global_step
        if cfg.sample_every and s>0 and s % cfg.sample_every == 0:
            try: show_samples(model)
            except Exception as e: print("sample skipped:", e)
        if cfg.eval_every and s>0 and s % cfg.eval_every == 0:
            try:
                loss, ppl = eval_loss_ppl(model)
                chrf = eval_translation_chrf(model)
                print(f"📊 [eval @ {s}] held-out loss={loss:.4f}  ppl={ppl:.2f}" +
                      (f"  translation chrF={chrf:.1f}" if chrf is not None else ""))
            except Exception as e: print("eval skipped:", e)
            gc.collect(); torch.cuda.empty_cache()

args = TrainingArguments(
    output_dir=cfg.out_dir,
    per_device_train_batch_size=cfg.micro_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    max_steps=cfg.max_steps,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type=cfg.lr_scheduler,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.grad_clip,
    bf16=cfg.bf16, fp16=not cfg.bf16,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps, save_total_limit=cfg.save_total_limit,
    gradient_checkpointing=cfg.gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=cfg.num_workers, dataloader_pin_memory=True,
    report_to="none", remove_unused_columns=False,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    seed=cfg.seed,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=collator, callbacks=[MonitorCallback()])
print("✅ trainer ready | max_steps:", cfg.max_steps)

## 10 · (Optional) Baseline eval — quality of the base model *before* fine-tuning

In [ ]:
loss, ppl = eval_loss_ppl(model)
chrf = eval_translation_chrf(model)
print(f"BEFORE training | held-out loss={loss:.4f}  ppl={ppl:.2f}  translation chrF={chrf:.1f}")
show_samples(model)

## 11 · TRAIN 🚀  (auto-resumes from the newest checkpoint)

In [ ]:
def newest_ckpt():
    if not cfg.resume: return None
    cks = sorted(Path(cfg.out_dir).glob("checkpoint-*"),
                 key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else 0)
    return str(cks[-1]) if cks else None

resume = newest_ckpt()
print("resuming from:", resume or "scratch")
trainer.train(resume_from_checkpoint=resume)

final = os.path.join(cfg.out_dir, "final")
trainer.save_model(final); tokenizer.save_pretrained(final)
with open(os.path.join(final, "grammora_config.json"), "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)
print("✅ training done — adapter saved to", final)

## 12 · Final evaluation — confirm you hit 90%+

In [ ]:
loss, ppl = eval_loss_ppl(model)
chrf = eval_translation_chrf(model)
print(f"AFTER training  | held-out loss={loss:.4f}  ppl={ppl:.2f}  translation chrF={chrf:.1f}")
print("(lower loss/ppl = better; chrF is 0-100, higher = better. Compare to cell 10 baseline.)\n")
show_samples(model, max_new_tokens=220)

### ⏱️ How long / how many steps?

`max_steps` (not epochs) controls length because the data is streamed. One epoch ≈
`total_examples / (micro_batch × grad_accum × num_GPUs)` steps. Your corpus ≈ **10.6M
examples**, so on **1 GPU** at the defaults (8×8) one epoch ≈ **165k steps** — far more
than `max_steps=12000`. That's fine: 12k steps already sees ~1.5M diverse, mixed examples
and usually lands 90%+ on these tasks. **Raise `max_steps` for more coverage; watch the
`chrF` and live samples and stop when they plateau.**

On more GPUs the effective batch grows, so **lower `max_steps` accordingly** (e.g. 48
GPUs → ~2–3k steps for a couple of epochs).

## 14 · Merge LoRA → standalone deployable model

In [ ]:
from peft import PeftModel
merged = os.path.join(cfg.out_dir, "merged")
base = AutoModelForCausalLM.from_pretrained(cfg.base_model, torch_dtype=torch.bfloat16, trust_remote_code=True)
m = PeftModel.from_pretrained(base, os.path.join(cfg.out_dir,"final")).merge_and_unload()
m.config.use_cache = True
m.save_pretrained(merged, safe_serialization=True)
tokenizer.save_pretrained(merged)
print("✅ standalone model saved to", merged)
print("serve:  python -m vllm.entrypoints.openai.api_server --model", merged)

## 15 · Chat — one model, every task

In [ ]:
@torch.no_grad()
def grammora(prompt, system=None, max_new_tokens=400, temperature=0.7):
    model.eval(); dev = next(model.parameters()).device
    msgs = ([{"role":"system","content":system or cfg.system_prompt}] if cfg.add_system_prompt else []) + \
           [{"role":"user","content":prompt}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(dev)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, top_p=0.9, repetition_penalty=1.1,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

for p in [
    "اس کی گرامر درست کریں۔ میں کل اسکول جاتا ہوں اور کتاب پڑھی تھی۔",
    "Summarize this. پاکستان اسٹاک مارکیٹ میں آج زبردست تیزی دیکھی گئی اور انڈیکس چار سو پوائنٹس بڑھ گیا۔",
    "Translate this into English. علم روشنی کی مانند ہے۔",
    "Translate this into Urdu. Knowledge is power.",
    "اس کا خلاصہ اور پیرا فریز دونوں کریں۔ محنت کامیابی کی کنجی ہے۔",
]:
    print("🟦", p)
    print("🟩", grammora(p), "\n")

## ✅ Tips to actually reach 90%+

1. **Use a strong base** — `Qwen2.5-7B-Instruct` is the sweet spot; `14B` raises the
   ceiling if you have VRAM/time.
2. **Train enough** — raise `max_steps` until held-out **loss/ppl plateaus** and
   translation **chrF** stops climbing (watch the `📊 [eval]` lines).
3. **Data quality > quantity** — the model copies your assistant answers. Clean targets
   ⇒ clean model.
4. **LoRA r=64 is near full-FT quality** for domain SFT and merges to a standalone model.
5. **Judge with the numbers, not vibes** — compare cell 12 (after) to cell 10 (before);
   the gap is your real improvement.